# 🍛 GlycoLens — YOLOv8 Indian Food & Reference Detector Fine-Tuning

This notebook fine-tunes **YOLOv8** on Indian meal classes and reference objects (*katori*, thali plate) using free Google Colab GPU (T4).

> **Hackathon Workflow**:
> 1. Runtime -> Change runtime type -> **T4 GPU**
> 2. Run all cells
> 3. Download `best.pt` and place into `glycolens/data/models/yolov8n_food.pt`

In [ ]:
# Step 1: Install Ultralytics and check GPU
!pip install ultralytics roboflow

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Download Indian Food Detection Dataset
# Classes: ['roti', 'rice', 'dal', 'paneer', 'rajma', 'chole', 'dosa', 'idli', 'katori', 'plate']
from roboflow import Roboflow

# Option A: Download curated Indian food dataset directly via Roboflow Universe
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("food-detection").project("indian-food-plates")
# dataset = project.version(1).download("yolov8")

# Option B: Create local data.yaml configuration
data_yaml = '''
path: ./indian_food_data
train: images/train
val: images/val

names:
  0: roti
  1: rice
  2: dal
  3: paneer
  4: rajma
  5: chole
  6: dosa
  7: idli
  8: katori
  9: plate
'''
with open('data.yaml', 'w') as f:
    f.write(data_yaml)
print("Created data.yaml")

In [ ]:
# Step 3: Fine-Tune YOLOv8 Nano on Indian Meal Classes
from ultralytics import YOLO

# Load lightweight YOLOv8 Nano pre-trained backbone
model = YOLO('yolov8n.pt')

# Train for 25 epochs (takes ~8 minutes on free Colab T4 GPU)
results = model.train(
    data='data.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='glycolens_yolo',
    name='indian_food_detector'
)

In [ ]:
# Step 4: Validate and Print Evaluation Metrics
metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

# Record these real values in glycolens/docs/evaluation.md under Section 2!

In [ ]:
# Step 5: Download the Trained Weights
from google.colab import files
files.download('glycolens_yolo/indian_food_detector/weights/best.pt')